# Mask review

Visual check of the mass masks in `data/processed/masks/` against the mammograms.
Build them first with `python -m mammo_seg.data.masks`. Outputs are stripped
before commit (`nbstripout`).

Each row is one mass: the whole image with the mask outline in red, then a zoomed
crop of the same lesion without and with the outline. The question for every row:
does the outline sit on the lesion, and does it follow its edge?


In [ ]:
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pydicom
from scipy import ndimage

from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.masks import mask_path, read_mask


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
RAW = ROOT / "data" / "raw" / "INbreast Release 1.0"
MANIFEST = ROOT / "data" / "processed" / "manifest.json"
MASKS = ROOT / "data" / "processed" / "masks"

## Masks and positive images

One case per connected region of a mask (usually one mass; a few images have two or
three). A few polygons also leave isolated specks of 1 to 47 pixels next to the main
region (spiky vertices in the annotation). They are part of the mask as drawn but
are not worth a row, so regions under `MIN_REGION` pixels are skipped here.


In [ ]:
manifest = read_manifest(MANIFEST)
positives = [image for image in manifest.images if image.has_mass]
missing = [
    image.file_id
    for image in manifest.images
    if not mask_path(MASKS, image.file_id).exists()
]
assert not missing, (
    f"masks not found, run python -m mammo_seg.data.masks: {missing[:5]}"
)
print(len(manifest.images), "images,", len(positives), "with a mass")


@lru_cache(maxsize=4)
def load_pixels(dicom_path: str) -> np.ndarray:
    return pydicom.dcmread(RAW / dicom_path).pixel_array


MIN_REGION = 100  # pixels
cases = []
specks = 0
for image in positives:
    mask = read_mask(mask_path(MASKS, image.file_id))
    regions, count = ndimage.label(mask)
    for index, box in enumerate(ndimage.find_objects(regions)):
        area = int((regions[box] == index + 1).sum())
        if area < MIN_REGION:
            specks += 1
        else:
            cases.append((image, box, area))
print(len(cases), "masks to review,", specks, "tiny specks skipped")

In [ ]:
STEP = 4  # downscale of the whole-image view, for display only
MARGIN = 0.5  # of the mask size, around the zoomed crop
MIN_CROP = 160  # pixels


def window(pixels: np.ndarray) -> tuple[float, float]:
    foreground = pixels[pixels > 0]
    low, high = np.percentile(foreground if foreground.size else pixels, [1, 99.5])
    return float(low), float(high)


def show(selected: list) -> None:
    fig, axes = plt.subplots(
        len(selected), 3, figsize=(16, 5.2 * len(selected)), squeeze=False
    )
    for (image, box, area), row in zip(selected, axes, strict=True):
        pixels = load_pixels(image.dicom_path)
        mask = read_mask(mask_path(MASKS, image.file_id))
        low, high = window(pixels)

        whole, outline, zoom_plain, zoom_outline = row[0], row[0], row[1], row[2]
        whole.imshow(
            np.clip((pixels[::STEP, ::STEP] - low) / (high - low), 0, 1), cmap="gray"
        )
        outline.contour(mask[::STEP, ::STEP], levels=[0.5], colors="red", linewidths=1)
        whole.set_title(
            f"{image.file_id}  {image.laterality} {image.view}  BI-RADS {image.birads}"
            f"  masses in image: {image.n_masses}",
            fontsize=9,
        )

        (r0, r1), (c0, c1) = [(s.start, s.stop) for s in box]
        pad = max(MIN_CROP, int(MARGIN * max(r1 - r0, c1 - c0)))
        r0, r1 = max(0, r0 - pad), min(pixels.shape[0], r1 + pad)
        c0, c1 = max(0, c0 - pad), min(pixels.shape[1], c1 + pad)
        crop = pixels[r0:r1, c0:c1].astype(np.float32)
        crop_low, crop_high = window(crop)
        shown = np.clip((crop - crop_low) / (crop_high - crop_low), 0, 1)
        zoom_plain.imshow(shown, cmap="gray")
        zoom_outline.imshow(shown, cmap="gray")
        zoom_outline.contour(
            mask[r0:r1, c0:c1], levels=[0.5], colors="red", linewidths=1
        )
        zoom_plain.set_title("zoom", fontsize=9)
        zoom_outline.set_title(f"zoom with the mask ({area} px)", fontsize=9)
        for axis in row:
            axis.axis("off")
    plt.tight_layout()
    plt.show()

## Go through the masks

Set `PAGE` and run the cell again to move on. `PAGE_SIZE` masks are shown at a time.


In [ ]:
PAGE = 0
PAGE_SIZE = 4

pages = -(-len(cases) // PAGE_SIZE)
print(f"page {PAGE + 1} of {pages}")
show(cases[PAGE * PAGE_SIZE : (PAGE + 1) * PAGE_SIZE])

### Show everything

Draws every page one after the other (long output; clear it afterwards).


In [ ]:
for page in range(pages):
    show(cases[page * PAGE_SIZE : (page + 1) * PAGE_SIZE])

## Quantitative check

Masses are denser than the tissue around them, so a mask in the right place is
brighter than a ring of tissue around it. The same test on a mask that is shifted
or flipped should be at chance level. This catches an offset, flip or swapped axes
for the whole set at once. It does not say how tightly the outline follows the edge.


In [ ]:
def contrast(pixels: np.ndarray, mask: np.ndarray) -> float:
    """Mean intensity inside the mask minus a ring around it (background excluded)."""
    inside = mask > 0
    if not inside.any():
        return float("nan")
    ring = ndimage.binary_dilation(inside, iterations=15)
    ring &= ~ndimage.binary_dilation(inside, iterations=3)
    ring &= pixels > 0
    if not ring.any():
        return float("nan")
    return float(pixels[inside].mean() - pixels[ring].mean())


variants = {
    "real mask": lambda mask: mask,
    "shifted sideways": lambda mask: np.roll(mask, 300 // STEP, axis=1),
    "shifted vertically": lambda mask: np.roll(mask, 300 // STEP, axis=0),
    "flipped vertically": lambda mask: mask[::-1],
}
results: dict[str, list[float]] = {name: [] for name in variants}
for image in positives:
    pixels = load_pixels(image.dicom_path)[::STEP, ::STEP].astype(np.float32)
    mask = read_mask(mask_path(MASKS, image.file_id))[::STEP, ::STEP]
    for name, transform in variants.items():
        results[name].append(contrast(pixels, transform(mask)))

for name, values in results.items():
    valid = np.array([v for v in values if not np.isnan(v)])
    print(
        f"{name:20} {len(valid):3d} images, brighter than the ring in "
        f"{100 * (valid > 0).mean():5.1f}%, median contrast {np.median(valid):7.1f}"
    )